# EDA 16 · Supervised learning on the manual segment

**Goal.** Train Logistic Regression, Random Forest and XGBoost exclusively on the
February `state='manual'` segment and compare against the unsupervised AE from EDA 15.

**Data splits (manual segment only):**
- **Train**: February fit split — NC + C (~83K users, ~31.78% churn)
- **Tune**: February tune split — threshold calibration (~14K users)
- **Test (OOT)**: March manual — all users, fully out-of-time

**Features (same as EDA 15):**
- CAT (1): `expiry_month`
- NUM (6): `longest_gap`, `latest_gap_ratio`, `renewal_frequency`, `active_days_90`, `active_days_30`, `active_weeks_90`

## 0. Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from datetime import date
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_curve, f1_score, precision_score, recall_score,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

EDA_DIR = next(p for p in (Path.cwd(), Path.cwd() / "eda") if (p / "_common.py").exists())
sys.path.insert(0, str(EDA_DIR))
import _common as C

FEB_CUTOFF = date(2017, 1, 31)
MAR_CUTOFF = date(2017, 2, 28)
SEED = C.SEED

CAT = ["expiry_month"]
NUM = [
    "longest_gap",
    "latest_gap_ratio",
    "renewal_frequency",
    "active_days_90", "active_days_30",
    "active_weeks_90",
]
FEATURES = CAT + NUM

print(f"features: {len(CAT)} cat + {len(NUM)} num = {len(FEATURES)} total")

## 1. Feature engineering

In [ ]:
def _expiry_month(df, cutoff):
    first_label = (pd.Timestamp(cutoff) + pd.Timedelta(days=1)).date()
    nxt   = (pd.Timestamp(first_label) + pd.offsets.MonthBegin(1)).date()
    after = (pd.Timestamp(nxt)         + pd.offsets.MonthBegin(1)).date()
    e = pl.col("last_expire")
    return df.with_columns(
        expiry_month=pl.when(pl.col("expiry_state") == "no txn").then(pl.lit("no txn"))
        .when(e.is_null()).then(pl.lit("unknown"))
        .when(e <= pl.lit(cutoff)).then(pl.lit("lapsed"))
        .when(e < pl.lit(nxt)).then(pl.lit("label month"))
        .when(e < pl.lit(after)).then(pl.lit("next month"))
        .otherwise(pl.lit("later"))
    )


def _full90_listening(users):
    return (
        C.clean_user_days(users)
        .with_columns(week=(pl.col("day") // 7))
        .group_by("msno")
        .agg(
            active_days_90=pl.len(),
            secs_90=pl.col("total_secs").sum(),
            active_weeks_90=pl.col("week").n_unique(),
        )
        .collect(engine="streaming")
    )


def build_features(users, cutoff, keep_methods):
    C.set_cutoff(cutoff)
    txn  = C.clean_transactions(users)
    st   = C.state_block(C.subscription_state(txn, users), keep_methods=keep_methods)
    base = (_expiry_month(st, cutoff)
            .join(C.history_block(txn, users), on="msno")
            .join(C.listening_block(users),    on="msno")
            .join(C.member_block(users),        on="msno")
            .with_columns(payer_days=pl.col("payer_days").clip(upper_bound=730)))

    full90 = _full90_listening(users)

    df = (base
          .join(full90, on="msno", how="left")
          .with_columns(
              pl.col("active_days_90").fill_null(0),
              pl.col("secs_90").fill_null(0.0),
              pl.col("active_weeks_90").fill_null(0),
          )
          .with_columns(
              renewal_frequency=(
                  pl.col("payer_days").cast(pl.Float64) /
                  (pl.col("n_lapses").cast(pl.Float64) + 1.0)
              ),
              secs_per_week_90=(
                  pl.col("secs_90").cast(pl.Float64) /
                  (pl.col("active_weeks_90").cast(pl.Float64) + 1.0)
              ),
              latest_gap_ratio=(
                  pl.col("longest_gap").cast(pl.Float64) /
                  (pl.col("active_days_90").cast(pl.Float64) + 1.0)
              ),
          ))

    C.set_cutoff(FEB_CUTOFF)
    return df


print("Feature engineering functions ready.")

## 2. Load data and split

In [ ]:
cohort = C.load_cohort()
C.set_cutoff(FEB_CUTOFF)
_fit_raw   = cohort.filter(pl.col("split") == "fit").select("msno", "is_churn")
_fit_state = C.subscription_state(C.clean_transactions(_fit_raw), _fit_raw)
KEEP_METHODS = (
    _fit_state.group_by("last_payment_method_id").len()
    .filter((pl.col("len") >= 1000) & pl.col("last_payment_method_id").is_not_null())
    ["last_payment_method_id"].to_list()
)

feb_df = build_features(cohort.select("msno", "is_churn"), FEB_CUTOFF, KEEP_METHODS)
feb_df = feb_df.with_columns(
    pl.Series("split", cohort["split"].to_numpy()),
    pl.Series("y",     cohort["is_churn"].to_numpy()),
)

mar_users = pl.read_parquet(C.PARQUET_DIR / "train_v2.parquet", columns=["msno", "is_churn"])
mar_df    = build_features(mar_users, MAR_CUTOFF, KEEP_METHODS)
mar_df    = mar_df.with_columns(pl.Series("y", mar_users["is_churn"].to_numpy()))

# ── Manual segment splits ─────────────────────────────────────────────────────
train_df = feb_df.filter((pl.col("state") == "manual") & (pl.col("split") == "fit"))
tune_df  = feb_df.filter((pl.col("state") == "manual") & (pl.col("split") == "tune"))
test_df  = mar_df.filter(pl.col("state") == "manual")

X_train = train_df.select(FEATURES).to_pandas()
y_train = train_df["y"].to_numpy()

X_tune  = tune_df.select(FEATURES).to_pandas()
y_tune  = tune_df["y"].to_numpy()

X_test  = test_df.select(FEATURES).to_pandas()
y_test  = test_df["y"].to_numpy()

print(f"{'Split':<8}  {'n':>8}  {'churn':>8}  {'churn%':>8}")
print("─" * 38)
for name, y in [("train", y_train), ("tune", y_tune), ("test", y_test)]:
    print(f"{name:<8}  {len(y):>8,}  {int(y.sum()):>8,}  {y.mean():>8.2%}")

## 3. Models

In [ ]:
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
    ]), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
], remainder="drop")

pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"scale_pos_weight (XGB): {pos_weight:.2f}")

MODELS = {
    "LR":  LogisticRegression(
        C=1.0, class_weight="balanced", max_iter=1000, random_state=SEED
    ),
    "RF":  RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_leaf=5,
        class_weight="balanced", random_state=SEED, n_jobs=-1,
    ),
    "XGB": XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        scale_pos_weight=pos_weight, random_state=SEED,
        eval_metric="logloss", verbosity=0,
    ),
}

pipelines = {}
for name, model in MODELS.items():
    print(f"Entrenando {name}...", end=" ", flush=True)
    pipe = Pipeline([("prep", preprocessor), ("clf", model)])
    pipe.fit(X_train, y_train)
    pipelines[name] = pipe
    print("listo")

## 4. Evaluation

In [ ]:
def _best_f1_thr(y, s):
    p, r, t = precision_recall_curve(y, s)
    denom   = p[:-1] + r[:-1]
    f1      = np.where(denom > 0, 2 * p[:-1] * r[:-1] / denom, 0.0)
    return float(t[np.argmax(f1)])


def _eval(y, proba, thr, model_name, label):
    yp = (proba >= thr).astype(int)
    return {
        "model":      model_name,
        "escenario":  label,
        "roc_auc":    float(roc_auc_score(y, proba)),
        "avg_prec":   float(average_precision_score(y, proba)),
        "f1":         float(f1_score(y, yp, zero_division=0)),
        "precision":  float(precision_score(y, yp, zero_division=0)),
        "recall":     float(recall_score(y, yp, zero_division=0)),
        "threshold":  round(thr, 4),
        "flag_rate":  float(yp.mean()),
    }


eval_rows = []
for name, pipe in pipelines.items():
    proba_tune = pipe.predict_proba(X_tune)[:, 1]
    proba_test = pipe.predict_proba(X_test)[:, 1]
    thr = _best_f1_thr(y_tune, proba_tune)
    eval_rows.append(_eval(y_tune, proba_tune, thr, name, "Feb · tune"))
    eval_rows.append(_eval(y_test, proba_test, thr, name, "Mar · manual"))

results_df = pd.DataFrame(eval_rows)
SHOW = ["model", "escenario", "roc_auc", "avg_prec", "f1", "precision", "recall", "flag_rate"]
print(results_df[SHOW].round(4).to_string(index=False))

## 5. Precision-Recall curves

In [ ]:
COLORS = {"LR": "steelblue", "RF": "seagreen", "XGB": "darkorange"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (X_ev, y_ev, title) in zip(axes, [
    (X_tune, y_tune, "Feb · tune  (threshold calibration)"),
    (X_test, y_test, "Mar · manual  (OOT)"),
]):
    baseline = float(y_ev.mean())
    for name, pipe in pipelines.items():
        proba = pipe.predict_proba(X_ev)[:, 1]
        p, r, _ = precision_recall_curve(y_ev, proba)
        ap = average_precision_score(y_ev, proba)
        ax.plot(r, p, color=COLORS[name], lw=1.8, label=f"{name}  AP={ap:.3f}")
    ax.axhline(baseline, color="gray", ls="--", lw=1, label=f"baseline ({baseline:.2%})")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(fontsize=9)

plt.suptitle("Precision-Recall — LR / RF / XGB · segmento manual", fontsize=12)
plt.tight_layout()
C.save_fig(fig, "16_pr_curves")
plt.show()

## 6. Resumen y comparación con EDA 15

In [ ]:
print("=" * 72)
print("Mar · manual (OOT) — modelos supervisados")
print("=" * 72)
mar_res = results_df[results_df["escenario"] == "Mar · manual"][SHOW].round(4)
print(mar_res.to_string(index=False))
print()
print("─" * 72)
print("Referencia — EDA 15 (AE/VAE unsupervised, mismas 7 features):")
print("  AE   roc_auc=0.6538  avg_prec=0.5797  f1=0.5970  prec=0.4569  rec=0.8612")
print("  VAE  roc_auc=0.5573  avg_prec=0.4985  f1=0.5789  prec=0.4074  rec=1.0000")
print()
print("Feature set:")
print(f"  CAT ({len(CAT)}): {CAT}")
print(f"  NUM ({len(NUM)}): {NUM}")